A smart search feature, a Semantic Search Engine. This is the technology behind Google Search and modern RAG pipelines.

# Step 1: Setting Up the Database

In [1]:
import pandas as pd

# Our "Database" of articles
data = [
    {'title': 'Python for Beginners', 'content': 'Learn the basics of Python programming, loops, and functions.'},
    {'title': 'Machine Learning 101', 'content': 'An introduction to neural networks and supervised learning.'},
    {'title': 'Healthy Cooking Tips', 'content': 'How to cook nutritious meals quickly using fresh ingredients.'},
    {'title': 'Best Laptops of 2026', 'content': 'Reviews of the top computing devices for work and gaming.'},
    {'title': 'The Future of AI', 'content': 'Discussing Large Language Models and the ethics of artificial intelligence.'},
    {'title': 'Yoga for Stress', 'content': 'Simple stretches and breathing exercises to relax.'}
]

df = pd.DataFrame(data)
print("Data loaded successfully!")
print(df.head())

Data loaded successfully!
                  title                                            content
0  Python for Beginners  Learn the basics of Python programming, loops,...
1  Machine Learning 101  An introduction to neural networks and supervi...
2  Healthy Cooking Tips  How to cook nutritious meals quickly using fre...
3  Best Laptops of 2026  Reviews of the top computing devices for work ...
4      The Future of AI  Discussing Large Language Models and the ethic...


# Step 2: Generating Embeddings

In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load the pre-trained model
# This will download about 80MB the first time you run it
print("Loading model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# Combine title and content for better context
# We want to search across both the headline and the body text
sentences = df['title'] + ': ' + df['content']

# Generate embeddings
print("Encoding data...")
embeddings = model.encode(sentences.tolist())

print(f"Shape of embeddings: {embeddings.shape}")

Loading model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding data...
Shape of embeddings: (6, 384)


The embeddings.shape output will likely be (6, 384).

6: We have 6 documents.
384: Each document is now a list of 384 numbers. These numbers are its coordinates on our map.

# Step 3: Indexing with FAISS

In [6]:
!pip install faiss-cpu
import faiss

# FAISS works with float32 type
embeddings = np.array(embeddings).astype("float32")

# Create the index
# 384 is the dimension of the vectors from the MiniLM model
index = faiss.IndexFlatL2(384)

# Add our embeddings to the index
index.add(embeddings)

print(f"Total documents in index: {index.ntotal}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 82.5 MB/s eta 0:00:00
Total documents in index: 6


# Step 4: The Search Function

connect the user to the data. When someone searches, we need to:

Convert their query into a vector (using the same model).
Ask FAISS to find the nearest neighbours.
Return the readable text.


In [7]:
def search(query, k=3):
    # 1. Encode the query
    query_vector = model.encode([query])

    # 2. Search the index
    # k is the number of results we want
    distances, indices = index.search(np.array(query_vector).astype("float32"), k)

    # 3. Format results
    results = []
    for i in range(k):
        idx = indices[0][i]
        # FAISS returns -1 if it can't find neighbors (unlikely here)
        if idx != -1:
            result = {
                'title': df.iloc[idx]['title'],
                'content': df.iloc[idx]['content'],
                'score': float(distances[0][i]) # Lower score = closer distance = better match
            }
            results.append(result)

    return results

# Step 5: Testing

In [8]:
# Test 1: Keyword mismatch
print("\n--- Test Query: 'computer for gaming' ---")
results = search("computer for gaming")
for res in results:
    print(f"Found: {res['title']}")

# Test 2: Conceptual search
print("\n--- Test Query: 'how to relax' ---")
results = search("how to relax")
for res in results:
    print(f"Found: {res['title']}")

# Test 3: Technical search
print("\n--- Test Query: 'coding tutorial' ---")
results = search("coding tutorial")
for res in results:
    print(f"Found: {res['title']}")


--- Test Query: 'computer for gaming' ---
Found: Best Laptops of 2026
Found: Python for Beginners
Found: Machine Learning 101

--- Test Query: 'how to relax' ---
Found: Yoga for Stress
Found: Healthy Cooking Tips
Found: Best Laptops of 2026

--- Test Query: 'coding tutorial' ---
Found: Python for Beginners
Found: Machine Learning 101
Found: The Future of AI


search engine that understands intent, not just words.

In [9]:
# Install Streamlit
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 55.7 MB/s eta 0:00:00


Now, let's create the Streamlit application file (`app.py`). This file will contain all the logic for our semantic search engine, including data loading, embedding generation, FAISS indexing, and the search function, along with the Streamlit UI components.

In [10]:
%%writefile app.py
import streamlit as st
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

st.set_page_config(layout='wide')

st.title('Semantic Search Engine')
st.markdown('Enter a query and find semantically similar articles.')

@st.cache_resource
def load_data_and_model():
    # Our "Database" of articles
    data = [
        {'title': 'Python for Beginners', 'content': 'Learn the basics of Python programming, loops, and functions.'},
        {'title': 'Machine Learning 101', 'content': 'An introduction to neural networks and supervised learning.'},
        {'title': 'Healthy Cooking Tips', 'content': 'How to cook nutritious meals quickly using fresh ingredients.'},
        {'title': 'Best Laptops of 2026', 'content': 'Reviews of the top computing devices for work and gaming.'},
        {'title': 'The Future of AI', 'content': 'Discussing Large Language Models and the ethics of artificial intelligence.'},
        {'title': 'Yoga for Stress', 'content': 'Simple stretches and breathing exercises to relax.'}
    ]

    df = pd.DataFrame(data)

    # Load the pre-trained model
    model = SentenceTransformer('all-MiniLM-L6-v2')

    # Combine title and content for better context
    sentences = df['title'] + ': ' + df['content']

    # Generate embeddings
    embeddings = model.encode(sentences.tolist())
    embeddings = np.array(embeddings).astype("float32")

    # Create the FAISS index
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)

    return df, model, index

df, model, index = load_data_and_model()

def search(query, k=3):
    # 1. Encode the query
    query_vector = model.encode([query])

    # 2. Search the index
    distances, indices = index.search(np.array(query_vector).astype("float32"), k)

    # 3. Format results
    results = []
    for i in range(k):
        idx = indices[0][i]
        if idx != -1:
            result = {
                'title': df.iloc[idx]['title'],
                'content': df.iloc[idx]['content'],
                'score': float(distances[0][i])
            }
            results.append(result)

    return results

query_input = st.text_input('Search for articles:', 'How to relax')

if query_input:
    st.subheader('Search Results:')
    search_results = search(query_input)
    if search_results:
        for i, res in enumerate(search_results):
            st.write(f"**{i+1}. {res['title']}** (Score: {res['score']:.4f})")
            st.write(res['content'])
            st.markdown('---')
    else:
        st.write('No results found.')


Writing app.py


After running the above cell, an `app.py` file will be created in your Colab environment. To run the Streamlit application, execute the following command in a new code cell. A public URL will be provided that you can click to access your Streamlit app.

In [11]:
!streamlit run app.py & npx localtunnel --port 8501

⠙⠹⠸⠼⠴

⠦⠧⠇⠏Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) 2026-07-04 19:30:11.164 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.185.4.170:8501



KeyboardInterrupt: 

In [12]:
%%writefile requirements.txt
streamlit
pandas
sentence-transformers
numpy
faiss-cpu

Writing requirements.txt
